Fact Table Block Structure

The sequence of independent cells is structured as follows:

Cell 1: Initial Load (Join Bronze Header + Bronze Item + Dimension key lookups against _stg).

Cleansing: ALPHA conversion on order keys and text standardization.

DQ1 (Mandatory Keys): Non-null SalesOrder and SalesOrderItem.

DQ2 (Format - HARD): OData date parsing (/Date(ms)/) and ISO currency code validation (3 characters).

DQ3 (Business Metrics): Quantities > 0 and consistent amounts.

DQ4 (Reconciliation - SOFT): Analytical comparison of $\sum \text{NetAmount}$ vs. HeaderTotalNetAmount logging warnings into DqWarnings.

DQ5 (Chronological Consistency and Technical Integrity): Validation of _rescued_data IS NULL.

Sweeper: Consolidated routing of QUARANTINED rows to sales_order_quarantine.

Transition to Ready State.

Final Promotion: Idempotent MERGE into fact_sales_order_item_stg.

In [0]:
%python
def run_block_0_extraction(batch_df):
    batch_df.createOrReplaceTempView("batch_bronze_items_raw")
    
    spark.sql("TRUNCATE TABLE workspace.silver.fact_sales_order_item_tmp;")
    
    spark.sql("""
        INSERT INTO workspace.silver.fact_sales_order_item_tmp (
            SalesOrder, SalesOrderItem, SoldToParty, Material,
            CustomerKey, ProductKey, OrganizationKey,
            CreationDateKey, SalesOrderDateKey, RequestedDeliveryDateKey,
            CreationDate_Raw, SalesOrderDate_Raw, RequestedDeliveryDate_Raw,
            SalesOrderItemCategory, DeliveryStatus, SDProcessStatus, BillingStatus,
            TransactionCurrency, RequestedQuantity, ConfdDeliveredQuantity,
            NetAmount, TaxAmount, CostAmount, GrossWeight, NetWeight, ItemVolume,
            HeaderTotalNetAmount, SourceSystem, IsSynthetic, IsDeleted,
            _rescued_data, _ingestion_timestamp, DqWarnings,
            Status_Cleansing, Status_DQ_Keys, Status_DQ_Format,
            Status_DQ_Metrics, Status_DQ_Dates, Status_DQ_Integrity, Status_Process
        )
        SELECT 
            i.SalesOrder, i.SalesOrderItem, h.SoldToParty, i.Material,
            coalesce(c.CustomerKey, cent_c.CustomerKey) AS CustomerKey,
            coalesce(p.ProductKey, cent_p.ProductKey) AS ProductKey,
            coalesce(o.OrganizationKey, cent_o.OrganizationKey) AS OrganizationKey,
            CAST(NULL AS INT), CAST(NULL AS INT), CAST(NULL AS INT),
            CAST(h.CreationDate AS STRING), CAST(h.SalesOrderDate AS STRING), CAST(h.RequestedDeliveryDate AS STRING),
            i.SalesOrderItemCategory, i.DeliveryStatus, i.SDProcessStatus, i.OrderRelatedBillingStatus, h.TransactionCurrency,
            CAST(i.RequestedQuantity AS DECIMAL(18, 3)), CAST(i.ConfdDelivQtyInOrderQtyUnit AS DECIMAL(18, 3)),
            CAST(i.NetAmount AS DECIMAL(18, 2)), CAST(i.TaxAmount AS DECIMAL(18, 2)), CAST(i.CostAmount AS DECIMAL(18, 2)),
            CAST(i.ItemGrossWeight AS DECIMAL(18, 3)), CAST(i.ItemNetWeight AS DECIMAL(18, 3)), CAST(i.ItemVolume AS DECIMAL(18, 3)),
            CAST(h.TotalNetAmount AS DECIMAL(18, 2)),
            coalesce(h.SourceSystem, i.SourceSystem, 'SAP_S4HANA'),
            coalesce(h.IsSynthetic, i.IsSynthetic, FALSE),
            CASE WHEN trim(coalesce(h.OverallSDDocumentRejectionSts, '')) = 'C' THEN TRUE ELSE FALSE END,
            coalesce(i._rescued_data, h._rescued_data),
            coalesce(i._ingestion_timestamp, h._ingestion_timestamp, current_timestamp()),
            ARRAY(), 'PENDING', 'PENDING', 'PENDING', 'PENDING', 'PENDING', 'PENDING', 'IN_PROGRESS'
        FROM (
            SELECT * FROM batch_bronze_items_raw
            QUALIFY ROW_NUMBER() OVER (PARTITION BY SalesOrder, SalesOrderItem ORDER BY _ingestion_timestamp DESC NULLS LAST) = 1
        ) i
        INNER JOIN (
            SELECT * FROM workspace.bronze.sales_order_header
            QUALIFY ROW_NUMBER() OVER (PARTITION BY SalesOrder ORDER BY _ingestion_timestamp DESC NULLS LAST) = 1
        ) h ON i.SalesOrder = h.SalesOrder
        LEFT JOIN workspace.silver.dim_customer_stg c ON coalesce(regexp_replace(h.SoldToParty, '^0+', ''), h.SoldToParty) = c.SoldToParty AND c.IsCurrent = TRUE
        LEFT JOIN workspace.silver.dim_customer_stg cent_c ON cent_c.SoldToParty = 'UNKNOWN' AND cent_c.IsCurrent = TRUE
        LEFT JOIN workspace.silver.dim_product_stg p ON coalesce(regexp_replace(i.Material, '^0+', ''), i.Material) = p.Material AND p.IsCurrent = TRUE
        LEFT JOIN workspace.silver.dim_product_stg cent_p ON cent_p.Material = 'UNKNOWN' AND cent_p.IsCurrent = TRUE
        LEFT JOIN workspace.silver.dim_organization_stg o ON h.SalesOrganization = o.SalesOrganization AND h.DistributionChannel = o.DistributionChannel AND h.OrganizationDivision = o.OrganizationDivision AND o.IsCurrent = TRUE
        LEFT JOIN workspace.silver.dim_organization_stg cent_o ON cent_o.SalesOrganization = 'UNKNOWN' AND cent_o.IsCurrent = TRUE;
    """)

In [0]:
%python
def run_cleansing():
    spark.sql("""
        UPDATE workspace.silver.fact_sales_order_item_tmp
        SET 
            SalesOrder = CASE 
                WHEN regexp_replace(SalesOrder, '^0+', '') = '' AND SalesOrder IS NOT NULL THEN '0'
                ELSE coalesce(regexp_replace(SalesOrder, '^0+', ''), SalesOrder)
            END,
            SalesOrderItem = CASE 
                WHEN regexp_replace(SalesOrderItem, '^0+', '') = '' AND SalesOrderItem IS NOT NULL THEN '0'
                ELSE coalesce(regexp_replace(SalesOrderItem, '^0+', ''), SalesOrderItem)
            END,
            SalesOrderItemCategory = trim(upper(SalesOrderItemCategory)),
            DeliveryStatus = trim(upper(DeliveryStatus)),
            SDProcessStatus = trim(upper(SDProcessStatus)),
            BillingStatus = trim(upper(BillingStatus)),
            TransactionCurrency = trim(upper(TransactionCurrency)),
            Status_Cleansing = 'COMPLETED'
        WHERE Status_Cleansing = 'PENDING' 
          AND Status_Process = 'IN_PROGRESS';
    """)

run_cleansing()

In [0]:
%python
def run_dq1():
    spark.sql("""
        UPDATE workspace.silver.fact_sales_order_item_tmp
        SET 
            -- 1. Evaluación de Claves de Negocio y Dimensionales
            Status_DQ_Keys = CASE 
                WHEN SalesOrder IS NULL OR trim(SalesOrder) = '' 
                  OR SalesOrderItem IS NULL OR trim(SalesOrderItem) = '' 
                  OR SoldToParty IS NULL OR trim(SoldToParty) = '' 
                  OR Material IS NULL OR trim(Material) = '' 
                  OR CustomerKey IS NULL 
                  OR ProductKey IS NULL 
                  OR OrganizationKey IS NULL 
                THEN 'FAILED'
                ELSE 'PASSED'
            END,
            
            -- 2. Transición inmediata a Cuarentena si falla alguna clave
            Status_Process = CASE 
                WHEN SalesOrder IS NULL OR trim(SalesOrder) = '' 
                  OR SalesOrderItem IS NULL OR trim(SalesOrderItem) = '' 
                  OR SoldToParty IS NULL OR trim(SoldToParty) = '' 
                  OR Material IS NULL OR trim(Material) = '' 
                  OR CustomerKey IS NULL 
                  OR ProductKey IS NULL 
                  OR OrganizationKey IS NULL 
                THEN 'QUARANTINED'
                ELSE Status_Process
            END
            
        WHERE Status_Cleansing = 'COMPLETED' 
          AND Status_Process = 'IN_PROGRESS';
    """)

run_dq1()

In [0]:
%python
def run_dq2():
    spark.sql(r"""
        UPDATE workspace.silver.fact_sales_order_item_tmp
        SET 
            -- 1. CreationDateKey derivation (YYYYMMDD)
            CreationDateKey = CAST(date_format(
                CASE 
                    WHEN CreationDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                        THEN try_cast(from_unixtime(CAST(regexp_extract(CreationDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                    ELSE try_cast(CreationDate_Raw AS TIMESTAMP)
                END, 'yyyyMMdd'
            ) AS INT),

            -- 2. SalesOrderDateKey derivation (YYYYMMDD)
            SalesOrderDateKey = CAST(date_format(
                CASE 
                    WHEN SalesOrderDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                        THEN try_cast(from_unixtime(CAST(regexp_extract(SalesOrderDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                    ELSE try_cast(SalesOrderDate_Raw AS TIMESTAMP)
                END, 'yyyyMMdd'
            ) AS INT),

            -- 3. RequestedDeliveryDateKey derivation (YYYYMMDD or -1 if null)
            RequestedDeliveryDateKey = coalesce(
                CAST(date_format(
                    CASE 
                        WHEN RequestedDeliveryDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                            THEN try_cast(from_unixtime(CAST(regexp_extract(RequestedDeliveryDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                        ELSE try_cast(RequestedDeliveryDate_Raw AS TIMESTAMP)
                    END, 'yyyyMMdd'
                ) AS INT),
                -1
            ),

            -- 4. Format Rule Evaluation (HARD)
            Status_DQ_Format = CASE 
                WHEN (
                    CASE 
                        WHEN CreationDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                            THEN try_cast(from_unixtime(CAST(regexp_extract(CreationDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                        ELSE try_cast(CreationDate_Raw AS TIMESTAMP)
                    END
                ) IS NULL
                OR (
                    CASE 
                        WHEN SalesOrderDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                            THEN try_cast(from_unixtime(CAST(regexp_extract(SalesOrderDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                        ELSE try_cast(SalesOrderDate_Raw AS TIMESTAMP)
                    END
                ) IS NULL
                OR (
                    RequestedDeliveryDate_Raw IS NOT NULL AND (
                        CASE 
                            WHEN RequestedDeliveryDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                                THEN try_cast(from_unixtime(CAST(regexp_extract(RequestedDeliveryDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                            ELSE try_cast(RequestedDeliveryDate_Raw AS TIMESTAMP)
                        END
                    ) IS NULL
                )
                OR TransactionCurrency IS NULL 
                OR NOT (TransactionCurrency RLIKE '^[A-Z]{3}$')
                THEN 'FAILED'
                ELSE 'PASSED'
            END,

            -- 5. Lifecycle Transition
            Status_Process = CASE 
                WHEN (
                    CASE 
                        WHEN CreationDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                            THEN try_cast(from_unixtime(CAST(regexp_extract(CreationDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                        ELSE try_cast(CreationDate_Raw AS TIMESTAMP)
                    END
                ) IS NULL
                OR (
                    CASE 
                        WHEN SalesOrderDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                            THEN try_cast(from_unixtime(CAST(regexp_extract(SalesOrderDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                        ELSE try_cast(SalesOrderDate_Raw AS TIMESTAMP)
                    END
                ) IS NULL
                OR (
                    RequestedDeliveryDate_Raw IS NOT NULL AND (
                        CASE 
                            WHEN RequestedDeliveryDate_Raw RLIKE '^/Date\\([0-9]+\\)/$' 
                                THEN try_cast(from_unixtime(CAST(regexp_extract(RequestedDeliveryDate_Raw, '^/Date\\(([0-9]+)\\)/$', 1) AS BIGINT) / 1000) AS TIMESTAMP)
                            ELSE try_cast(RequestedDeliveryDate_Raw AS TIMESTAMP)
                        END
                    ) IS NULL
                )
                OR TransactionCurrency IS NULL 
                OR NOT (TransactionCurrency RLIKE '^[A-Z]{3}$')
                THEN 'QUARANTINED'
                ELSE Status_Process
            END

        WHERE Status_Cleansing = 'COMPLETED' 
          AND Status_DQ_Keys = 'PASSED'
          AND Status_Process = 'IN_PROGRESS';
    """)

run_dq2()

In [0]:
%python
def run_dq3():
    spark.sql("""
        UPDATE workspace.silver.fact_sales_order_item_tmp
        SET 
            Status_DQ_Metrics = CASE 
                WHEN RequestedQuantity IS NULL OR RequestedQuantity <= 0
                  OR NetAmount IS NULL OR NetAmount < 0
                  OR (TaxAmount IS NOT NULL AND TaxAmount < 0)
                  OR (CostAmount IS NOT NULL AND CostAmount < 0)
                  OR (GrossWeight IS NOT NULL AND GrossWeight < 0)
                  OR (NetWeight IS NOT NULL AND NetWeight < 0)
                  OR (ItemVolume IS NOT NULL AND ItemVolume < 0)
                THEN 'FAILED'
                ELSE 'PASSED'
            END,
            Status_Process = CASE 
                WHEN RequestedQuantity IS NULL OR RequestedQuantity <= 0
                  OR NetAmount IS NULL OR NetAmount < 0
                  OR (TaxAmount IS NOT NULL AND TaxAmount < 0)
                  OR (CostAmount IS NOT NULL AND CostAmount < 0)
                  OR (GrossWeight IS NOT NULL AND GrossWeight < 0)
                  OR (NetWeight IS NOT NULL AND NetWeight < 0)
                  OR (ItemVolume IS NOT NULL AND ItemVolume < 0)
                THEN 'QUARANTINED'
                ELSE Status_Process
            END
        WHERE Status_Process = 'IN_PROGRESS' 
          AND Status_DQ_Format = 'PASSED';
    """)

run_dq3()

In [0]:
%python
def run_dq4():
    spark.sql("""
        MERGE INTO workspace.silver.fact_sales_order_item_tmp AS tgt
        USING (
            SELECT 
                SalesOrder,
                SalesOrderItem,
                CASE 
                    WHEN abs(coalesce(HeaderTotalNetAmount, 0) - sum(coalesce(NetAmount, 0)) OVER (PARTITION BY SalesOrder)) > 0.01 
                    THEN TRUE 
                    ELSE FALSE 
                END AS is_mismatch
            FROM workspace.silver.fact_sales_order_item_tmp
            WHERE Status_Process = 'IN_PROGRESS'
        ) AS src
        ON tgt.SalesOrder = src.SalesOrder 
        AND tgt.SalesOrderItem = src.SalesOrderItem
        WHEN MATCHED AND src.is_mismatch = TRUE THEN
            UPDATE SET 
                tgt.DqWarnings = array_append(tgt.DqWarnings, 'RULE_8_QUADRATURE_MISMATCH');
    """)

run_dq4()

In [0]:
%python
def run_dq5():
    spark.sql("""
        UPDATE workspace.silver.fact_sales_order_item_tmp
        SET 
            -- 1. Temporal Consistency Validation
            Status_DQ_Dates = CASE 
                WHEN RequestedDeliveryDateKey IS NOT NULL 
                  AND CreationDateKey IS NOT NULL 
                  AND RequestedDeliveryDateKey < CreationDateKey THEN 'FAILED'
                ELSE 'PASSED'
            END,
            
            -- 2. Zero Technical Corruption Validation
            Status_DQ_Integrity = CASE 
                WHEN _rescued_data IS NOT NULL THEN 'FAILED'
                ELSE 'PASSED'
            END,
            
            -- 3. Lifecycle Transition to Quarantine if any fails
            Status_Process = CASE 
                WHEN (RequestedDeliveryDateKey IS NOT NULL AND CreationDateKey IS NOT NULL AND RequestedDeliveryDateKey < CreationDateKey)
                  OR _rescued_data IS NOT NULL THEN 'QUARANTINED'
                ELSE Status_Process
            END

        WHERE Status_Process = 'IN_PROGRESS'
          AND Status_DQ_Metrics = 'PASSED';
    """)

run_dq5()

In [0]:
%python
def run_quarantine_routing():
    spark.sql("""
        INSERT INTO workspace.silver.sales_order_quarantine (
            SourceEntity,
            RecordIdentifier,
            FailedRule,
            FailureSeverity,
            RawRecord,
            IngestionTimestamp
        )
        SELECT 
            'FACT_SALES_ORDER_ITEM' AS SourceEntity,
            concat(coalesce(SalesOrder, 'UNKNOWN_ORDER'), '-', coalesce(SalesOrderItem, 'UNKNOWN_ITEM')) AS RecordIdentifier,
            CASE 
                WHEN Status_DQ_Keys = 'FAILED' THEN 'DQ_KEYS_NULL_OR_EMPTY'
                WHEN Status_DQ_Format = 'FAILED' THEN 'DQ6_FORMAT_OR_CURRENCY_INVALID'
                WHEN Status_DQ_Metrics = 'FAILED' THEN 'DQ_METRICS_OUT_OF_BOUNDS'
                WHEN Status_DQ_Dates = 'FAILED' THEN 'DQ_DATES_CHRONOLOGY_MISMATCH'
                WHEN Status_DQ_Integrity = 'FAILED' THEN 'DQ5_RESCUED_DATA_CORRUPT'
                ELSE 'UNKNOWN_FAILURE'
            END AS FailedRule,
            'HARD' AS FailureSeverity,
            to_json(named_struct(
                'SalesOrder', SalesOrder,
                'SalesOrderItem', SalesOrderItem,
                'SoldToParty', SoldToParty,
                'Material', Material,
                'CustomerKey', CustomerKey,
                'ProductKey', ProductKey,
                'OrganizationKey', OrganizationKey,
                'RequestedQuantity', RequestedQuantity,
                'NetAmount', NetAmount,
                'TransactionCurrency', TransactionCurrency,
                'CreationDate_Raw', CreationDate_Raw,
                'SalesOrderDate_Raw', SalesOrderDate_Raw,
                'RequestedDeliveryDate_Raw', RequestedDeliveryDate_Raw,
                '_rescued_data', _rescued_data
            )) AS RawRecord,
            current_timestamp() AS IngestionTimestamp
        FROM workspace.silver.fact_sales_order_item_tmp
        WHERE Status_Process = 'QUARANTINED';
    """)

run_quarantine_routing()

In [0]:
%python
def run_transition_status():
    spark.sql("""
        UPDATE workspace.silver.fact_sales_order_item_tmp
        SET Status_Process = 'READY_FOR_STG'
        WHERE Status_Process = 'IN_PROGRESS'
          AND Status_DQ_Keys = 'PASSED'
          AND Status_DQ_Format = 'PASSED'
          AND Status_DQ_Metrics = 'PASSED'
          AND Status_DQ_Dates = 'PASSED'
          AND Status_DQ_Integrity = 'PASSED';
    """)

run_transition_status()

In [0]:
%python
def run_promotion():
    spark.sql("""
        -- Final promotion with idempotent MERGE
        MERGE INTO workspace.silver.fact_sales_order_item_stg AS tgt
        USING (
            SELECT 
                SalesOrder,
                SalesOrderItem,
                CustomerKey,
                ProductKey,
                OrganizationKey,
                coalesce(CreationDateKey, -1) AS CreationDateKey,
                coalesce(SalesOrderDateKey, -1) AS SalesOrderDateKey,
                coalesce(RequestedDeliveryDateKey, -1) AS RequestedDeliveryDateKey,
                SalesOrderItemCategory,
                DeliveryStatus,
                SDProcessStatus,
                BillingStatus,
                TransactionCurrency,
                RequestedQuantity,
                ConfdDeliveredQuantity,
                NetAmount,
                TaxAmount,
                CostAmount,
                GrossWeight,
                NetWeight,
                ItemVolume,
                SourceSystem,
                IsSynthetic,
                coalesce(IsDeleted, FALSE) AS IsDeleted,
                DqWarnings,
                _ingestion_timestamp
            FROM workspace.silver.fact_sales_order_item_tmp
            WHERE Status_Process = 'READY_FOR_STG'
        ) AS src
        ON tgt.SalesOrder = src.SalesOrder 
        AND tgt.SalesOrderItem = src.SalesOrderItem

        WHEN MATCHED THEN
            -- Update metrics, statuses, and warnings if line item already exists
            UPDATE SET 
                tgt.CustomerKey = src.CustomerKey,
                tgt.ProductKey = src.ProductKey,
                tgt.OrganizationKey = src.OrganizationKey,
                tgt.CreationDateKey = src.CreationDateKey,
                tgt.SalesOrderDateKey = src.SalesOrderDateKey,
                tgt.RequestedDeliveryDateKey = src.RequestedDeliveryDateKey,
                tgt.SalesOrderItemCategory = src.SalesOrderItemCategory,
                tgt.DeliveryStatus = src.DeliveryStatus,
                tgt.SDProcessStatus = src.SDProcessStatus,
                tgt.BillingStatus = src.BillingStatus,
                tgt.TransactionCurrency = src.TransactionCurrency,
                tgt.RequestedQuantity = src.RequestedQuantity,
                tgt.ConfdDeliveredQuantity = src.ConfdDeliveredQuantity,
                tgt.NetAmount = src.NetAmount,
                tgt.TaxAmount = src.TaxAmount,
                tgt.CostAmount = src.CostAmount,
                tgt.GrossWeight = src.GrossWeight,
                tgt.NetWeight = src.NetWeight,
                tgt.ItemVolume = src.ItemVolume,
                tgt.SourceSystem = src.SourceSystem,
                tgt.IsSynthetic = src.IsSynthetic,
                tgt.IsDeleted = src.IsDeleted,
                tgt.DqWarnings = src.DqWarnings,
                tgt._ingestion_timestamp = src._ingestion_timestamp

        WHEN NOT MATCHED THEN
            -- Insert new line item
            INSERT (
                SalesOrder,
                SalesOrderItem,
                CustomerKey,
                ProductKey,
                OrganizationKey,
                CreationDateKey,
                SalesOrderDateKey,
                RequestedDeliveryDateKey,
                SalesOrderItemCategory,
                DeliveryStatus,
                SDProcessStatus,
                BillingStatus,
                TransactionCurrency,
                RequestedQuantity,
                ConfdDeliveredQuantity,
                NetAmount,
                TaxAmount,
                CostAmount,
                GrossWeight,
                NetWeight,
                ItemVolume,
                SourceSystem,
                IsSynthetic,
                IsDeleted,
                DqWarnings,
                _ingestion_timestamp
            )
            VALUES (
                src.SalesOrder,
                src.SalesOrderItem,
                src.CustomerKey,
                src.ProductKey,
                src.OrganizationKey,
                src.CreationDateKey,
                src.SalesOrderDateKey,
                src.RequestedDeliveryDateKey,
                src.SalesOrderItemCategory,
                src.DeliveryStatus,
                src.SDProcessStatus,
                src.BillingStatus,
                src.TransactionCurrency,
                src.RequestedQuantity,
                src.ConfdDeliveredQuantity,
                src.NetAmount,
                src.TaxAmount,
                src.CostAmount,
                src.GrossWeight,
                src.NetWeight,
                src.ItemVolume,
                src.SourceSystem,
                src.IsSynthetic,
                src.IsDeleted,
                src.DqWarnings,
                src._ingestion_timestamp
            );
    """)

run_promotion()

In [0]:
%python
CHECKPOINT_PATH = "s3://dataheib-s3-bucket/checkpoints/silver_fact_sales_order_item/"

def process_silver_fact_microbatch(batch_df, batch_id):
    if batch_df.isEmpty():
        print(f"[Microbatch {batch_id}] Vacío. Omitiendo.")
        return

    print(f"[Microbatch {batch_id}] Procesando {batch_df.count()} registros...")
    
    # Invocación en secuencia de cada celda del notebook
    run_block_0_extraction(batch_df)
    run_cleansing()
    run_dq1()
    run_dq2()
    run_dq3()
    run_dq4()
    run_dq5()
    run_quarantine_routing()
    run_transition_status()
    run_promotion()
    
    print(f"[Microbatch {batch_id}] Integración completada con éxito.")

# Ejecución incremental controlada
(
    spark.readStream
    .table("workspace.bronze.sales_order_item")
    .writeStream
    .foreachBatch(process_silver_fact_microbatch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)